In [ ]:
!pip install -q --upgrade bitsandbytes==0.48.2 trl==0.25.1 accelerate peft wandb datasets huggingface_hub
%env CUDA_VISIBLE_DEVICES=0

In [ ]:
import os
import torch
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset
import wandb
from peft import PeftModel, LoraConfig
from trl import SFTTrainer, SFTConfig

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
wandb_api_key = user_secrets.get_secret("WANDB_API_KEY")

login(hf_token, add_to_git_credential=True)

os.environ["WANDB_API_KEY"] = wandb_api_key
wandb.login()

PROJECT_NAME = "price"
os.environ["WANDB_PROJECT"] = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "false"
os.environ["WANDB_WATCH"] = "false"

In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
HF_USER = "sairuthwik112"

DATA_USER = "ed-donner"
DATASET_NAME = f"{DATA_USER}/items_prompts_full"

# The repo holding your step-1500 adapter weights
PREVIOUS_ADAPTER_REPO = f"{HF_USER}/price-2026-07-15_15.18.55-full"

# New run name for this continuation phase
RUN_NAME = "2026-07-19_continue-from-1500"
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

BATCH_SIZE = 8
MAX_SEQUENCE_LENGTH = 512
GRADIENT_ACCUMULATION_STEPS = 4

# Remaining steps to reach your original 2500 target
REMAINING_STEPS = 1000

LEARNING_RATE = 1e-4     # lower than the original 2e-4 — avoids destabilizing already-learned weights
WARMUP_RATIO = 0.0       # no warmup needed — weights are already warmed up from the first phase
LR_SCHEDULER_TYPE = 'cosine'
WEIGHT_DECAY = 0.01
OPTIMIZER = "paged_adamw_32bit"

capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8

LOG_STEPS = 50
SAVE_STEPS = 200
VAL_SIZE = 5000
print("loaded")

In [ ]:
dataset = load_dataset(DATASET_NAME)
train = dataset['train'].shuffle(seed=123)   # reshuffle so the continuation doesn't just repeat the exact same first 1500-steps' order
val = dataset['val'].select(range(VAL_SIZE))

print(f"Train size: {len(train)}")
print(f"Val size: {len(val)}")

wandb.init(project=PROJECT_NAME, name=RUN_NAME)

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map={"": 0},
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

# Load your existing step-1500 LoRA weights on top of the base model — this is the warm start
model = PeftModel.from_pretrained(base_model, PREVIOUS_ADAPTER_REPO, is_trainable=True)

print(f"Loaded warm-started adapter from: {PREVIOUS_ADAPTER_REPO}")
print(f"Memory footprint: {model.get_memory_footprint() / 1e6:.1f} MB")

In [ ]:
from transformers import DataCollatorForLanguageModeling

class QwenTextCollator:
    def __init__(self, tokenizer):
        self.base_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    def __call__(self, features):
        cleaned_features = []
        for feature in features:
            cleaned = {k: v for k, v in feature.items() if k in ["input_ids", "attention_mask", "labels"]}
            cleaned_features.append(cleaned)
        return self.base_collator(cleaned_features)

collator = QwenTextCollator(tokenizer)

train_parameters = SFTConfig(
    output_dir=PROJECT_RUN_NAME,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim=OPTIMIZER,
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    logging_steps=LOG_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    fp16=not use_bf16,
    bf16=use_bf16,
    max_grad_norm=1.0,
    max_steps=REMAINING_STEPS,     # 1000 — the remaining portion of your original 2500 target
    warmup_ratio=WARMUP_RATIO,
    group_by_length=True,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    report_to="wandb",
    run_name=RUN_NAME,
    max_length=MAX_SEQUENCE_LENGTH,
    save_strategy="steps",
    hub_strategy="every_save",
    push_to_hub=True,
    hub_model_id=HUB_MODEL_NAME,
    hub_private_repo=True,
    eval_strategy="steps",
    eval_steps=SAVE_STEPS,
    dataset_text_field="prompt",
    remove_unused_columns=True
)

# No peft_config here — we're continuing training on the adapter already attached in Cell 5
fine_tuning = SFTTrainer(
    model=model,
    train_dataset=train,
    eval_dataset=val,
    args=train_parameters,
    processing_class=tokenizer,
    data_collator=collator
)

In [ ]:
fine_tuning.train()

fine_tuning.model.push_to_hub(PROJECT_RUN_NAME, private=True)
print(f"Saved to the hub: {PROJECT_RUN_NAME}")

wandb.finish()

In [ ]:
import glob

checkpoint_lookups = glob.glob(f"./{PROJECT_RUN_NAME}/checkpoint-*", recursive=True)
CHECKPOINT_PATH = sorted(checkpoint_lookups, key=lambda p: int(p.split("-")[-1]))[-1]
print(f"Final checkpoint: {CHECKPOINT_PATH}")

final_model = PeftModel.from_pretrained(base_model, CHECKPOINT_PATH)
final_model.push_to_hub("price-prediction-qwen-final", private=True)
print("Done — final model uploaded.")